### Inleveren

Lever in op CodeGrade onder **1.3 Graphs and Trees**

### Oefening 1 (Boom representatie)
- Maak een simpele boom met geneste dictionaries.
- Hoe zou je door deze boom kunnen lopen met code (eg alle elementen printen)?

In [23]:
dictionary = {
    "a": {"b": {"c": 1}, "d": 2},
    "e": 3,
    "f": {
        "g": {"h": {"i": 4}, "j": 5},
    },
}

def traverse_dict(d, depth=0):
    for key, value in d.items():
        print((depth * "  ") + ("\---" if depth > 0 else "") + str(key))
        if isinstance(value, dict):
            traverse_dict(value, depth + 1)
        

traverse_dict(dictionary)

a
  \---b
    \---c
  \---d
e
f
  \---g
    \---h
      \---i
    \---j


### Oefening 2 (Binaire boom)

- Maak een binaire boom: maak een class Node en verbind links & rechts.

In [25]:
class Node:
    def __init__(self, value = None, left = None, right = None):
        self.value = value
        self.left = left
        self.right = right

node = Node(5)

node.left = Node(3)
node.right = Node(7)

print(node.value)        # 5
print(node.left.value)   # 3
print(node.right.value)  # 7

5
3
7


### Oefening 3 — Volledige Binaire Boom

- Voor deze en volgende oefeningen kun je de boom uit Oefening 2 gebruiken.

- Bouw een boom waarbij elke node 0 of 2 kinderen heeft.

- Controleer dit met een functie.

In [68]:
# https://external-content.duckduckgo.com/iu/?u=http%3A%2F%2Fwww.btechsmartclass.com%2Fdata_structures%2Fds_images%2FBST%2520Example.png&f=1&nofb=1&ipt=46a63754cf420e61f748181bf972411a7693a9778f2374205ab46206f2d08dbd

node5 = Node(5)
node8 = Node(8, node5)
node15 = Node(15)

node10 = Node(10, node8, node15)
# node10 = Node(10)
node22 = Node(22)

node30 = Node(30)
node40 = Node(40)

node20 = Node(20, node10, node22)
node30 = Node(36, node30, node40)

tree = Node(25, node20, node30)

def traverse_tree(node, depth=0):
    if node is not None:
        print((depth * "  ") + ("\\---" if depth > 0 else "") + str(node.value))

        if (node.left is None) != (node.right is None):
            raise Exception("This tree is not complete, cannot traverse it.")
        
        traverse_tree(node.left, depth + 1)
        traverse_tree(node.right, depth + 1)


# traverse_tree(tree)


### Oefening 4 — Volmaakte Binaire Boom

- Schrijf een functie die checkt of hoogtes links en rechts gelijk zijn

In [56]:
def check_perfect_tree(node):
    if node is None:
        return True, 0  # (is_perfect, height)
    
    # Leaf node
    if node.left is None and node.right is None:
        return True, 0
    
    # Check left and right subtrees
    left_perfect, left_height = check_perfect_tree(node.left)
    right_perfect, right_height = check_perfect_tree(node.right)
    

    is_perfect = left_perfect and right_perfect and (left_height == right_height)
    height = max(left_height, right_height) + 1
    
    return is_perfect, height

result, height = check_perfect_tree(tree)
print(f"Is perfect: {result}, Height: {height}")


Is perfect: True, Height: 2


### Oefening 5 — BST insert en search

- Voeg getallen in een BST in en zoek er één.

In [61]:
def search_tree(node, value, path=[], direction=None):
    if node is None:
        return False, []
    
    current_path = path + [direction] if direction is not None else path
    
    if node.value == value:
        return True, current_path
    elif value < node.value:
        return search_tree(node.left, value, current_path, "left")
    else:
        return search_tree(node.right, value, current_path, "right")

found, path = search_tree(tree, 22)
print(f"Found: {found}, Path: {path}")

found, path = search_tree(tree, 30)
print(f"Found: {found}, Path: {path}")

found, path = search_tree(tree, 99)
print(f"Found: {found}, Path: {path}")


Found: True, Path: ['left', 'right']
Found: True, Path: ['right', 'left']
Found: False, Path: []


### Oefening 6 — Balanced Tree check

- Check of een boom gebalanceerd is (hoogteverschil ≤ 1).

In [69]:
def balanced_tree(node):
    if node is None:
        return True, 0  # (is_balanced, height)
    
    left_balanced, left_height = balanced_tree(node.left)
    right_balanced, right_height = balanced_tree(node.right)

    is_balanced = left_balanced and right_balanced and abs(left_height - right_height) <= 1
    height = max(left_height, right_height) + 1
    
    return is_balanced, height

result, height = balanced_tree(tree)
print(f"Is balanced: {result}, Height: {height}")

Is balanced: False, Height: 5


## Oefening 7 - Dijkstra

Gegeven de volgende graaf:

In [ ]:
A ---2--- B ---4--- D
| \        \         \
3   ---1--   2         1
|          \   \         \
C -----3-------- E ---5--- F


We gebruiken de volgende representatie voor deze graph:

In [70]:
graph = {
    'A': {'B': 2, 'C': 3, 'E': 1},
    'B': {'A': 2, 'D': 4, 'E': 2},
    'C': {'A': 3, 'E': 3},
    'D': {'B': 4, 'F': 1},
    'E': {'A': 1, 'B': 2, 'C': 3, 'F': 5},
    'F': {'D': 1, 'E': 5}
}

start = 'A'
goal = 'F'

Welke knopen zijn volgens Dijkstra het kortste te bereiken vanaf A? Implementeer.

Een mogelijke opzet zou zijn:
- Alle afstanden beginnen op oneindig
- afstand startknop op 0
- Zoek de knoop met kleinste afstand die nog niet is bezocht
- Voeg hem toe aan bezochte nodes
- Update de gewichten van alle buren van de node

- je mag heapq import gebruiken

In [76]:
import heapq as hq


def dijkstra_distances(graph, start):
    distances = {}

    heap = [(0, start)]

    while heap:
        dist, node = hq.heappop(heap)
        if node in distances:
            continue
        distances[node] = dist
        for neighbor, weight in graph[node].items():
            if neighbor not in distances:
                hq.heappush(heap, (dist + weight, neighbor))

    return distances

distances = dijkstra_distances(graph, start)
print(f"Distances from {start}: {distances}")

Distances from A: {'A': 0, 'E': 1, 'B': 2, 'C': 3, 'D': 6, 'F': 6}


In [ ]:
def dijkstra_goal(graph, start, goal):
    heap = [(0, start)]
    visited = set()

    while heap:
        dist, node = hq.heappop(heap)
        if node in visited:
            continue
        visited.add(node)

        if node == goal:
            return dist

        for neighbor, weight in graph[node].items():
            if neighbor not in visited:
                hq.heappush(heap, (dist + weight, neighbor))

    return float("inf")


distance_to_goal = dijkstra_goal(graph, start, goal)
print(f"Distance from {start} to {goal}: {distance_to_goal}")

Distance from A to F: 6


## Oefening 8 - A*

Gebruik dezelfde graph, maar dan nu met A* algoritme. Gebruik geen externe libraries (behalve heapq). Maak een functie die de afstand vindt van A naar F zoekt.

Implementeer 2 heurestieken en test ze:
- Manhattan
- Euclidean

Je kunt aannemen dat de punten in een grid staan:

In [ ]:
coords = {"A": (0, 0), "B": (2, 0), "C": (0, 3), "D": (6, 0), "E": (1, 2), "F": (7, 1)}


def manhattan_distance(coord1, coord2):
    return abs(coord1[0] - coord2[0]) + abs(coord1[1] - coord2[1])


def euclidean_distance(coord1, coord2):
    return ((coord1[0] - coord2[0]) ** 2 + (coord1[1] - coord2[1]) ** 2) ** 0.5


def a_star(graph, coords, start, goal, heuristic=manhattan_distance):
    heap = [(0, start)]
    visited = set()
    g_costs = {start: 0}  # Actual distance from start

    while heap:
        f_cost, node = hq.heappop(heap)
        if node in visited:
            continue
        visited.add(node)

        if node == goal:
            return g_costs[goal]

        for neighbor, weight in graph[node].items():
            if neighbor not in visited:
                new_g_cost = g_costs[node] + weight
                if neighbor not in g_costs or new_g_cost < g_costs[neighbor]:
                    g_costs[neighbor] = new_g_cost
                    h_cost = heuristic(coords[neighbor], coords[goal])
                    f_cost = new_g_cost + h_cost
                    hq.heappush(heap, (f_cost, neighbor))

    return float("inf")


distance = a_star(graph, coords, start, goal, manhattan_distance)
print(f"A* (Manhattan) distance from {start} to {goal}: {distance}")

distance = a_star(graph, coords, start, goal, euclidean_distance)
print(f"A* (Euclidean) distance from {start} to {goal}: {distance}")

A* (Manhattan) distance from A to F: 7
A* (Euclidean) distance from A to F: 6


## Oefening 9 BFS

Probeer met BFS algoritme node F te vinden vanuit A. Gebruik een queue in de oplossing.
Tip: je kunt bijvoorbeeld pop(0) gebruiken op een lijst om een queue te simuleren

In [83]:
def bfs(graph, start, goal):
    queue = [start]
    visited = set()
    distances = {start: 0}
    
    while queue:
        node = queue.pop(0)  # FIFO - breadth first
        
        if node in visited:
            continue
        visited.add(node)
        
        if node == goal:
            return True, distances[goal]
        
        for neighbor in graph[node].keys():
            if neighbor not in visited and neighbor not in distances:
                distances[neighbor] = distances[node] + 1
                queue.append(neighbor)
    
    return False, float("inf")

found, distance = bfs(graph, start, goal)
print(f"BFS - Found: {found}, Distance from {start} to {goal}: {distance}")


BFS - Found: True, Distance from A to F: 2


## Oefening 10 DFS

Probeer met DFS algoritme node F te vinden vanuit A. Gebruik een stack in de oplossing.
Tip: je kunt bijvoorbeeld een gewone lijst gebruiken op een stack te simuleren

In [84]:
def dfs(graph, start, goal, visited=None, distance=0):
    if visited is None:
        visited = set()
    
    if start == goal:
        return True, distance
    
    visited.add(start)
    
    for neighbor in graph[start].keys():
        if neighbor not in visited:
            found, dist = dfs(graph, neighbor, goal, visited, distance + 1)
            if found:
                return True, dist
    
    return False, float("inf")

found, distance = dfs(graph, start, goal)
print(f"DFS - Found: {found}, Distance from {start} to {goal}: {distance}")

DFS - Found: True, Distance from A to F: 3
